In [155]:
#!pip install xarray
#!pip install scipy
#!pip install cdsapi

#### Import Libraries

In [156]:
import xarray as xr
import numpy as np
import pandas as pd
import marineHeatWaves as mhw
import datetime as dt
import math

import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import from_origin

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import griddata
from matplotlib.gridspec import GridSpec

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

#### Define Slice Function

The coordinates have to be defined in 0-360 format.

In [157]:
def spatial_subset(ds):
    out = ds.sel(lon = slice(350.0, 355.0),
                 lat = slice(36.0, 42.3))
    return out

In [158]:
from shapely.geometry import LineString
from shapely.ops import split


gdf = gpd.read_file('ICES_areas/ICES_Areas_20160601_cut_dense_3857.shp')
z9a = gdf[gdf['SubArea']== '9'][gdf['Division'] == 'a']
z9a = z9a.to_crs(epsg= 4256)

# Get polygon bounds
minx, miny, maxx, maxy = z9a.total_bounds

# Horizontal and vertical cut coordinates
cut1 = 39.35      # latitude for horizontal cut
cut2 = -8.93      # longitude for vertical cut
sagres_lat = 37.01  # latitude of Sagres

# Define cut lines
line1 = LineString([(minx, cut1), (maxx, cut1)])          # horizontal cut
line2 = LineString([(cut2, miny), (cut2, sagres_lat)])   # vertical cut up to Sagres

# Split polygon horizontally
geom = z9a.geometry.iloc[0]
parts1 = split(geom, line1)

# Split each horizontal part vertically up to Sagres
parts2 = []
for p in parts1.geoms:
    # Only split if polygon overlaps vertical line
    if p.bounds[0] < cut2 < p.bounds[2]:
        parts2.extend(split(p, line2).geoms)
    else:
        parts2.append(p)

# Build GeoDataFrame
z9a_3 = gpd.GeoDataFrame(geometry=parts2[1:], crs=z9a.crs)

/opt/anaconda3/envs/MHW/lib/python3.13/site-packages/geopandas/geodataframe.py:1891: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [159]:

def build_shape_mask(ds, shapefile):
    """
    Build a boolean mask on the native OISST grid.
    Assumes ds.lon is in 0-360 format.
    """

    gdf = shapefile.to_crs("EPSG:4326")

    # ---- convert shapefile lon from -180..180 → 0..360 ----
    gdf["geometry"] = gdf.geometry.translate(xoff=360)
    gdf["geometry"] = gdf.geometry.buffer(0)

    # grid spacing
    dlat = float(abs(ds.lat[1] - ds.lat[0]))
    dlon = float(abs(ds.lon[1] - ds.lon[0]))

    # raster transform (cell-centered grid)
    transform = from_origin(
        ds.lon.min().item() - dlon / 2,
        ds.lat.max().item() + dlat / 2,
        dlon,
        dlat
    )

    mask = geometry_mask(
        gdf.geometry,
        out_shape=(ds.dims["lat"], ds.dims["lon"]),
        transform=transform,
        invert=True
    )

    # flip latitude to match xarray ordering
    mask = mask[::-1, :]

    return xr.DataArray(
        mask,
        coords={"lat": ds.lat, "lon": ds.lon},
        dims=("lat", "lon")
    )


In [160]:
ref_ds = xr.open_dataset("oisst_daily/sst.day.mean.1982.nc")
shape_masks = {'NW': build_shape_mask(ref_ds, z9a_3.loc[[0]]),
               'SW': build_shape_mask(ref_ds, z9a_3.loc[[1]]),
               'S': build_shape_mask(ref_ds, z9a_3.loc[[2]])}

/var/folders/wq/98_by44j0113pwx7jtjgd74w0000gn/T/ipykernel_85493/4192309722.py:27: FutureWarning:

The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.

/var/folders/wq/98_by44j0113pwx7jtjgd74w0000gn/T/ipykernel_85493/4192309722.py:27: FutureWarning:

The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.

/var/folders/wq/98_by44j0113pwx7jtjgd74w0000gn/T/ipykernel_85493/4192309722.py:27: FutureWarning:

The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.



## Get Baseline

Baseline used is 1981-2010

In [161]:
baseline_df = pd.DataFrame()

for year in range(1982, 2012):

    var = xr.open_dataset(f"oisst_daily/sst.day.mean.{year}.nc", 
                        engine="netcdf4", decode_times=True)
    
    print(f'Opening {year}')
    
    var_sel = spatial_subset(var)
    
    var_sst = var_sel["sst"]

    baseline_df = pd.concat([baseline_df, var_sst.to_dataframe()])

baseline_df.reset_index(inplace= True)   
del var

Opening 1982
Opening 1983
Opening 1984
Opening 1985
Opening 1986
Opening 1987
Opening 1988
Opening 1989
Opening 1990
Opening 1991
Opening 1992
Opening 1993
Opening 1994
Opening 1995
Opening 1996
Opening 1997
Opening 1998
Opening 1999
Opening 2000
Opening 2001
Opening 2002
Opening 2003
Opening 2004
Opening 2005
Opening 2006
Opening 2007
Opening 2008
Opening 2009
Opening 2010
Opening 2011


In [162]:
## Conversion to ordinal time
def to_ordinal(time_array):
    ordinals = []

    for t in time_array:
        # Case 1: numpy.datetime64
        if isinstance(t, np.datetime64):
            ts = pd.Timestamp(t)
            ordinals.append(ts.to_pydatetime().date().toordinal())

        # Case 2: cftime objects
        else:
            ordinals.append(dt.date(t.year, t.month, t.day).toordinal())

    return np.array(ordinals)

## Detect MHWs

In [163]:
class Year:
    def __init__(self, year, shape_masks):
        self.year = year
        self.shape_masks = shape_masks
        
        self.sliced = {}
        for zone in ['NW', 'SW', 'S']:
            self.sliced[zone] = self.slice(zone)

        self.results = {}
        for zone in ['NW', 'SW', 'S']:
            self.results[zone] = self.detect_MHW(zone)
        
        self.table = self.create_table()


    def slice(self, zone):
        print('Slicing OISST Data')
        oisst = xr.open_dataset(f"oisst_daily/sst.day.mean.{self.year}.nc", 
                        engine="netcdf4", decode_times=True)
    
        oisst_sel = spatial_subset(oisst).where(self.shape_masks[zone])
        oisst_sst = oisst_sel["sst"]
        df = oisst_sst.to_dataframe().reset_index()
        df = df.dropna(subset=['sst'])
        return df
    
    
    def detect_MHW(self, zone, climatology_period=(1982, 2011)):
        print('Detecting MHWs')
        results = []

        for (lat, lon), group_year in self.sliced[zone].groupby(["lat", "lon"]):

            # --- year data ---
            y_sst = group_year["sst"].values
            y_time = group_year["time"].values
            y_time_ord = to_ordinal(y_time)

            # --- baseline for same grid cell ---
            base_group = baseline_df[
                (baseline_df["lat"] == lat) &
                (baseline_df["lon"] == lon)
            ]

            if base_group.empty:
                continue

            base_sst = base_group["sst"].values
            base_time = base_group["time"].values
            base_time_ord = to_ordinal(base_time)

            base_years = pd.to_datetime(base_time).year

            if (base_years.min() > climatology_period[0] or
                base_years.max() < climatology_period[1]):
                continue

            # --- MHW detection ---
            mhw_ev, clim = mhw.detect(
                y_time_ord,
                y_sst,
                climatologyPeriod= list(climatology_period),
                alternateClimatology= [base_time_ord, base_sst]
            )

            results.append({
                "year": pd.to_datetime(y_time[0]).year,
                "lat": lat,
                "lon": lon,
                "n_events": mhw_ev["n_events"],
                "total_duration": (
                    np.sum(mhw_ev["duration"])
                ),
                "max_intensity": (
                    np.max(mhw_ev["intensity_max"])
                    if mhw_ev["n_events"] > 0 else np.nan
                ),
                "mean_intensity":
                    float(np.mean(mhw_ev["intensity_mean"])),
                'cummulative_intensity': 
                    np.sum(mhw_ev['intensity_cumulative']),
            })

        return pd.DataFrame(results)
    

    def create_table(self):
        print('Creating Table')
        nw = self.results['NW'].drop(columns=['year', 'lat', 'lon']).agg(['min', 'max', 'mean', 'std'])
        sw = self.results['SW'].drop(columns=['year', 'lat', 'lon']).agg(['min', 'max', 'mean', 'std'])
        s = self.results['S'].drop(columns=['year', 'lat', 'lon']).agg(['min', 'max', 'mean', 'std'])

        final_df = pd.concat([nw, sw, s], keys=['NW', 'SW', 'S']).T
        return final_df

In [164]:
yearDict = {}
for i in range(1982, 2026):
    print(f'Treating {i}')
    year = Year(i, shape_masks)
    yearDict[i] = year

np.save('yearDict9a.npy', yearDict)


Treating 1982
Slicing OISST Data
Slicing OISST Data
Slicing OISST Data
Detecting MHWs
Detecting MHWs
Detecting MHWs
Creating Table
Treating 1983
Slicing OISST Data
Slicing OISST Data
Slicing OISST Data
Detecting MHWs
Detecting MHWs
Detecting MHWs
Creating Table
Treating 1984
Slicing OISST Data
Slicing OISST Data
Slicing OISST Data
Detecting MHWs
Detecting MHWs
Detecting MHWs
Creating Table
Treating 1985
Slicing OISST Data
Slicing OISST Data
Slicing OISST Data
Detecting MHWs
Detecting MHWs
Detecting MHWs
Creating Table
Treating 1986
Slicing OISST Data
Slicing OISST Data
Slicing OISST Data
Detecting MHWs
Detecting MHWs
Detecting MHWs
Creating Table
Treating 1987
Slicing OISST Data
Slicing OISST Data
Slicing OISST Data
Detecting MHWs
Detecting MHWs
Detecting MHWs
Creating Table
Treating 1988
Slicing OISST Data
Slicing OISST Data
Slicing OISST Data
Detecting MHWs
Detecting MHWs
Detecting MHWs
Creating Table
Treating 1989
Slicing OISST Data
Slicing OISST Data
Slicing OISST Data
Detecting MH

In [165]:
yearDictS = np.load('yearDict9a.npy', allow_pickle=True).item()

In [184]:
writer = pd.ExcelWriter('Summary Tables/9a.xlsx')
for year, obj in yearDictS.items():
    obj.table.to_excel(writer, sheet_name= str(year))
writer.close()

In [166]:
import plotly.express as px
import plotly.graph_objects as go

In [167]:
all_years = []

for year, obj in yearDictS.items():
    table = obj.table.copy()

    df_year = (
        table
        .stack(level=[0, 1], future_stack=True)
        .reset_index()
    )

    df_year.columns = ['Metric', 'Region', 'Agg', 'Value']
    df_year['Year'] = year

    all_years.append(df_year)

df = pd.concat(all_years, ignore_index=True)
df

,Metric,Region,Agg,Value,Year
0,n_events,NW,min,0.000000,1982
1,n_events,NW,max,0.000000,1982
2,n_events,NW,mean,0.000000,1982
3,n_events,NW,std,0.000000,1982
4,n_events,SW,min,0.000000,1982
...,...,...,...,...,...
2635,cummulative_intensity,SW,std,47.248855,2025
2636,cummulative_intensity,S,min,88.799168,2025
2637,cummulative_intensity,S,max,277.937802,2025
2638,cummulative_intensity,S,mean,199.184364,2025


## Graphs

In [185]:
fig = px.area(
    df[(df['Metric'] == 'n_events') & (df['Agg'] == 'mean')],
    x='Year',
    y='Value',
    facet_row= 'Region',
    color='Region',
    markers=True,
    title='Mean Number of Events - ICES 9a'
)

fig.update_layout(width=1000, height=600)

fig.show()


In [186]:
fig = px.area(
    df[(df['Metric'] == 'cummulative_intensity') & (df['Agg'] == 'mean')],
    x='Year',
    y='Value',
    facet_row= 'Region',
    color='Region',
    markers=True,
    title='Mean Cummulative Intensity - 9a'
)

fig.update_layout(width=1000, height=600)

fig.show()


In [187]:
fig = px.area(
    df[(df['Metric'] == 'total_duration') & (df['Agg'] == 'mean')],
    x='Year',
    y='Value',
    facet_row='Region',
    color= 'Region',
    markers=True,
    title='Mean MHW days - ICES 9a')

fig.update_layout(width=1000, height=600)

fig.show()
